In [2]:
%pip install ogb
%pip install EoN
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
%pip install torch_geometric



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 3.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for EoN: filename=EoN-1.2-py3-none-any.whl size=122468 sha256=c7083ebbdb2fd0f97b8ad3a34e1de83cade4e4de16cd91b4e5d11f08cee02251
  Stored in directory: /root/.cache/pip/wheels/53/e8/9d/8cce28ba7bdd58b1859da5582e7ca593ceae00cf9c996a3467
Successfully built EoN
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.1 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


# Config

In [3]:
config = {
    "undirected": False,

    "model": {
        "emb_dim": 256,
        "num_layers": [8],
        "dropout": 0.2,

        "conv_type": ["dir_gatv2"],
        "num_heads": 2,

        # Multiple pooling strategies
        "pool_types": ["attention"],

        "use_residual": True,
        "use_graph_norm": True,
    },

    "train": {
        "batch_size": 32,
        "epochs": 4,
        "lr": 1e-4,
        "weight_decay": 1e-5,
        "grad_clip": 1.0,
    },

    "data": {
        "max_depth": 20,
        "num_workers": 4,
    },

    "exp": {
        "seed": 42,
        "device": "cuda",
        "log_every": 50,
    }
}

# Data Loading

In [4]:
from ogb.graphproppred import PygGraphPropPredDataset
from torch_geometric.data import DataLoader
import torch
import torch_geometric

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])


def to_undirected_transform(data):
    data = data.clone()
    data.edge_index = torch_geometric.utils.to_undirected(
        data.edge_index,
        num_nodes=data.num_nodes,
    )
    return data


def uses_directed_edges(conv_types):
    if isinstance(conv_types, str):
        conv_types = [conv_types]
    return any(str(conv).startswith("dir_") for conv in conv_types)


def build_split_loaders(dataset_obj, split_idx_obj, batch_size, shuffle_train=True):
    train_loader_local = DataLoader(
        dataset_obj[split_idx_obj["train"]],
        batch_size=batch_size,
        shuffle=shuffle_train,
    )
    valid_loader_local = DataLoader(
        dataset_obj[split_idx_obj["valid"]],
        batch_size=batch_size,
        shuffle=False,
    )
    test_loader_local = DataLoader(
        dataset_obj[split_idx_obj["test"]],
        batch_size=batch_size,
        shuffle=False,
    )
    return train_loader_local, valid_loader_local, test_loader_local


d_name = "ogbg-code2"
dataset_directed = PygGraphPropPredDataset(name=d_name)
dataset_undirected = PygGraphPropPredDataset(name=d_name, transform=to_undirected_transform)

split_idx = dataset_directed.get_idx_split()

train_loader_directed, valid_loader_directed, test_loader_directed = build_split_loaders(
    dataset_directed,
    split_idx,
    batch_size=config["train"]["batch_size"],
)
train_loader_undirected, valid_loader_undirected, test_loader_undirected = build_split_loaders(
    dataset_undirected,
    split_idx,
    batch_size=config["train"]["batch_size"],
)

train_uses_directed = uses_directed_edges(config["model"].get("conv_type", []))
if train_uses_directed:
    train_loader = train_loader_directed
    valid_loader = valid_loader_directed
    test_loader = test_loader_directed
else:
    train_loader = train_loader_undirected
    valid_loader = valid_loader_undirected
    test_loader = test_loader_undirected

dataset = dataset_directed
print(f"Training/Eval loaders set to: {'directed' if train_uses_directed else 'undirected'} edges")

Downloaded 0.91 GB: 100%|██████████| 934/934 [00:18<00:00, 50.87it/s]


Extracting dataset/code2.zip


Processing...


Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 452741/452741 [00:01<00:00, 340893.89it/s]


...

100%|██████████| 452741/452741 [00:16<00:00, 28039.38it/s]


Saving...


Done!


Training/Eval loaders set to: directed edges


/tmp/ipykernel_55/2788204673.py:29: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader_local = DataLoader(
/tmp/ipykernel_55/2788204673.py:34: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  valid_loader_local = DataLoader(
/tmp/ipykernel_55/2788204673.py:39: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader_local = DataLoader(


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
import os
node_attr_path = os.path.join(os.curdir,dataset.root,"mapping","attridx2attr.csv.gz")
type_idx_path = os.path.join(os.curdir,dataset.root,"mapping","typeidx2type.csv.gz")

In [7]:
import os
import pandas as pd

node_attr_mapping  =  pd.read_csv(node_attr_path)
type_idx_mapping = pd.read_csv(type_idx_path)
node_type_vocab_size = len(type_idx_mapping)

In [8]:
new_rows = pd.DataFrame([{'attr idx': 10030, 'attr': '<pad>'},{'attr idx':10031,'attr':'<sos>'},{'attr idx':10032,'attr':'<eos>'}])
node_attr_mapping = pd.concat([node_attr_mapping, new_rows], ignore_index=True)

In [9]:
node_attr_to_idx_dict = {}
node_idx_to_attr_dict = {}
for i in range(len(node_attr_mapping)):
  node_attr_to_idx_dict[node_attr_mapping.iloc[i]['attr']] = node_attr_mapping.iloc[i]["attr idx"]
  node_idx_to_attr_dict[node_attr_mapping.iloc[i]["attr idx"]] = node_attr_mapping.iloc[i]["attr"]


# Encoder

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import (
    GINConv, GCNConv, GATv2Conv,DirGNNConv,
    GraphNorm,
    global_add_pool, global_mean_pool, global_max_pool,
    GlobalAttention,
    Set2Set # Added Set2Set import
)
class MultiHeadGlobalAttention(nn.Module):
    def __init__(self, emb_dim, heads=4):
        super().__init__()
        self.heads = heads

        self.gate_nn = nn.ModuleList([
            nn.Sequential(
                nn.Linear(emb_dim, 2*emb_dim),
                nn.LayerNorm(2*emb_dim),
                nn.ReLU(),
                nn.Linear(2*emb_dim, 1)
            )
            for _ in range(heads)
        ])

    def forward(self, x, batch):
        out = []

        for gate in self.gate_nn:
            scores = gate(x)                       # [N, 1]
            weights = torch_geometric.utils.softmax(scores, batch)
            pooled = torch_geometric.nn.global_add_pool(x * weights, batch)
            out.append(pooled)

        return torch.cat(out, dim=-1)

# ---------- Conv Factory ----------


def build_conv(conv_type, emb_dim, heads=2):
    if conv_type == "gin":
        mlp = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim)
        )
        return GINConv(nn=mlp, train_eps=True)

    elif conv_type == "dir_gatv2":
        base_conv = GATv2Conv(
            emb_dim,
            emb_dim ,
            heads,
            concat=False
        )
        return DirGNNConv(base_conv)
        
    elif conv_type == "dir_gin":
        mlp = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim)
        )
        base_conv = GINConv(nn=mlp, train_eps=True)
        base_conv.in_channels = emb_dim
        base_conv.out_channels = emb_dim
        return DirGNNConv(base_conv)
        
    elif conv_type == "dir_gcn":
        base_conv = GCNConv(emb_dim,emb_dim)
        return DirGNNConv(base_conv)

    elif conv_type == "gcn":
        return GCNConv(emb_dim, emb_dim)

    elif conv_type == "gatv2":
        assert emb_dim % heads == 0
        return GATv2Conv(
            emb_dim,
            emb_dim//2,
            heads=heads,
            concat=True
        )

    else:
        raise ValueError(f"Unknown conv type: {conv_type}")

def build_pool(pool_type, emb_dim, use_multi_stat=False, attn_heads=2):

    if pool_type == 'multi_stat':
        def pool(x, batch):
            return torch.cat([
                global_mean_pool(x, batch),
                global_max_pool(x, batch),
                global_add_pool(x, batch)
            ], dim=-1)
        return pool, emb_dim * 3

    if pool_type == "add":
        return lambda x, batch: global_add_pool(x, batch), emb_dim

    elif pool_type == "mean":
        return lambda x, batch: global_mean_pool(x, batch), emb_dim

    elif pool_type == "attention":
        if attn_heads == 1:
            gate_nn = nn.Sequential(
                nn.Linear(emb_dim, 2*emb_dim),
                nn.LayerNorm(2*emb_dim),
                nn.ReLU(),
                nn.Linear(2*emb_dim, 1)
            )
            return GlobalAttention(gate_nn), emb_dim
        else:
            pool = MultiHeadGlobalAttention(emb_dim, heads=attn_heads)
            return pool, emb_dim * attn_heads

    elif pool_type == "set2set":
        pool = Set2Set(emb_dim, processing_steps=3)
        return pool, emb_dim * 2

    else:
        raise ValueError(f"Unknown pool type: {pool_type}")

# ---------- Encoder ----------
class Encoder(nn.Module):
    def __init__(
        self,
        type_id_vocab_size,
        attr_id_vocab_size,
        max_depth,
        emb_dim,
        num_layers=3,
        conv_type="gin",
        pool_type="mean",
        dropout=0.2,
        num_heads=2,
        use_virtual_node=False,
        use_depth = True,
        use_residual = True,
        use_graph_norm = True
    ):
        super().__init__()

        self.emb_dim = emb_dim
        self.max_depth = max_depth
        self.num_layers = num_layers
        self.dropout = dropout
        self.use_virtual_node = use_virtual_node
        self.use_depth = use_depth
        self.use_residual = use_residual
        self.use_graph_norm = use_graph_norm

        # ---- Embeddings ----
        self.type_emb = nn.Embedding(type_id_vocab_size, emb_dim)
        self.attr_emb = nn.Embedding(attr_id_vocab_size, emb_dim)
        if use_depth:
          self.depth_emb = nn.Embedding(max_depth + 1, emb_dim)

          self.input_proj = nn.Linear(3 * emb_dim, emb_dim)
        else:
          self.input_proj = nn.Linear(2 * emb_dim, emb_dim)

        # ---- Conv stack ----
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(num_layers):
            self.convs.append(build_conv(conv_type, emb_dim, num_heads))
            self.norms.append(GraphNorm(emb_dim))

        # ---- Virtual Node ----
        if use_virtual_node:
            self.virtual_node_emb = nn.Embedding(1, emb_dim)
            self.virtual_mlp = nn.Sequential(
                nn.Linear(emb_dim, emb_dim),
                nn.ReLU(),
                nn.Linear(emb_dim, emb_dim)
            )

        # ---- Pooling ----
        self.pool, self.pool_dim = build_pool(
            pool_type, emb_dim
        )
        self.fc_out = nn.Sequential(nn.Linear(self.pool_dim,emb_dim*2),nn.ReLU(),nn.LayerNorm(emb_dim*2),nn.Linear(emb_dim*2,emb_dim),nn.LayerNorm(emb_dim))

    def forward(self, x, edge_index, depth, batch):
        # ---- Unpack ----
        type_ids = x[:, 0]
        attr_ids = x[:, 1]
        node_depth = torch.clamp(depth.squeeze(-1), 0, self.max_depth)

        # ---- Embed ----
        x1 = self.type_emb(type_ids)
        x2 = self.attr_emb(attr_ids)
        if self.use_depth:
          x3 = self.depth_emb(node_depth)
          x = torch.cat([x1, x2, x3], dim=1)
        else:
          x = torch.cat([x1, x2], dim=1)
        x = self.input_proj(x)

        # ---- Init virtual node ----
        if self.use_virtual_node:
            num_graphs = batch.max().item() + 1
            virtual_node = self.virtual_node_emb.weight.repeat(num_graphs, 1)

        # ---- Message Passing ----
        for layer, (conv, norm) in enumerate(zip(self.convs, self.norms)):

            # Add virtual node to all nodes
            if self.use_virtual_node:
                x = x + virtual_node[batch]

            h = conv(x, edge_index)
            h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)

            if self.use_residual:
                x = x + h
            if self.use_graph_norm:
                x = norm(x)

            # Update virtual node
            if self.use_virtual_node:
                pooled = global_add_pool(x, batch)
                virtual_node = virtual_node + self.virtual_mlp(pooled)

        # ---- Pooling ----
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.pool(x, batch)
        x = self.fc_out(x)

        return x

# Decoder

In [11]:
import torch
import torch.nn as nn
import math

class GraphConditionedTransformerDecoder(nn.Module):
    def __init__(
        self,
        embedding,
        vocab_size,
        emb_dim,
        pad_token_id,
        num_layers=4,
        nhead=4,
        max_len=20,
        dropout=0.1
    ):
        super().__init__()

        self.emb_dim = emb_dim
        self.embedding = embedding
        self.pad_token_id = pad_token_id

        # Positional embedding
        self.pos_emb = nn.Embedding(max_len, emb_dim)

        self.graph_ln = nn.LayerNorm(emb_dim)

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=nhead,
            batch_first=True,
            dropout=dropout
        )

        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers)

        # Output projection (weight tying, no bias)
        self.fc_out = nn.Linear(emb_dim, vocab_size, bias=False)
        self.fc_out.weight = self.embedding.weight

    def forward(self, graph_emb, tgt):
        """
        graph_emb: [B, H]
        tgt: [B, T]
        """

        B, T = tgt.shape

        # Token embeddings
        tok_emb = self.embedding(tgt) * math.sqrt(self.emb_dim)

        pos = torch.arange(T, device=tgt.device).unsqueeze(0)
        pos_emb = self.pos_emb(pos)

        x = tok_emb + pos_emb  # [B, T, H]

        graph_token = graph_emb.unsqueeze(1)  # [B, 1, H] 

        x = torch.cat([graph_token, x], dim=1)  # [B, T+1, H]

        seq_len = x.size(1)

        mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device),
            diagonal=1
        )
        mask = mask.masked_fill(mask == 1, float('-inf')).masked_fill(mask == 0, 0.0)

        mask[:, 0] = 0

        padding_mask = (tgt == self.pad_token_id)  # [B, T]

        graph_pad = torch.zeros((B, 1), dtype=torch.bool, device=tgt.device)

        padding_mask = torch.cat([graph_pad, padding_mask], dim=1)  # [B, T+1]

        x = self.transformer(
            x,
            mask=mask,
            src_key_padding_mask=padding_mask
        )

        # Remove graph token
        x = x[:, 1:, :]  # [B, T, H]

        logits = self.fc_out(x)

        return logits

# Graph2Seq

In [12]:
import torch
import torch.nn as nn

class Graph2SeqModel(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, data, tgt):
        graph_emb = self.encoder(
            data.x.to(device),
            data.edge_index.to(device),
            data.node_depth.to(device),
            data.batch.to(device)
        )

        logits = self.decoder(graph_emb, tgt)

        return logits

    def generate(self, data, max_tokens):
        graph_emb = self.encoder(
            data.x.to(device),
            data.edge_index.to(device),
            data.node_depth.to(device),
            data.batch.to(device)
        )

        num_graphs_in_batch = data.num_graphs

        tgt = torch.tensor([[10031] for i in range(num_graphs_in_batch)], device=device)

        for i in range(max_tokens):
            logits = self.decoder(graph_emb, tgt)
            next_token = torch.argmax(logits[:, -1, :], dim=1)
            tgt = torch.cat([tgt, next_token.unsqueeze(-1)], dim=1)

        return tgt

## Error Analysis by Graph Topology

In [13]:
from collections import defaultdict
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from ogb.graphproppred import Evaluator
from torch_geometric.data import DataLoader
from torch_geometric.utils import degree
import networkx as nx
from tqdm import tqdm

PAD = 10030
SOS = 10031
EOS = 10032
UNK = 10029

# Explicit checkpoints for topology analysis.
analysis_checkpoint_names = [
    "/kaggle/input/models/shreyasarun29/ogbg-code2/pytorch/default/3/8_dir_gatv2_attention.pt",
    "/kaggle/input/models/shreyasarun29/ogbg-code2/pytorch/default/3/8_dir_gcn_attention.pt",
    "/kaggle/input/models/shreyasarun29/ogbg-code2/pytorch/default/3/8_dir_gin_attention.pt",
    "/kaggle/input/models/shreyasarun29/ogbg-code2/pytorch/default/3/8_gcn_attention.pt",
    "/kaggle/input/models/shreyasarun29/ogbg-code2/pytorch/default/3/8_gin_attention.pt",
    "/kaggle/input/models/shreyasarun29/ogbg-code2/pytorch/default/3/8_gatv2_attention.pt",
    
]
analysis_checkpoint_glob = None  # Set to a glob like "*.pt" if you want auto-discovery.
analysis_max_tokens = 20
analysis_min_split_graphs = 16
analysis_output_dir = Path("error_analysis_outputs")
analysis_output_dir.mkdir(parents=True, exist_ok=True)

# Topology split bins (inclusive lower bound, exclusive upper bound except final bin).
NODE_COUNT_BINS = [
    (1, 11, "1-10"),
    (11, 21, "11-20"),
    (21, 36, "21-35"),
    (36, 51, "36-50"),
    (51, 76, "51-75"),
    (76, 101, "76-100"),
    (101, 151, "101-150"),
    (151, 201, "151-200"),
    (201, float("inf"), "200+"),
]
DEPTH_BINS = [
    (0, 4, "0-3"),
    (4, 7, "4-6"),
    (7, 10, "7-9"),
    (10, 13, "10-12"),
    (13, 16, "13-15"),
    (16, 19, "16-18"),
    (19, 22, "19-21"),
    (22, float("inf"), "22+"),
]
AVG_DEG_BINS = [
    (0.0, 0.5, "0-0.5"),
    (0.5, 1.0, "0.5-1"),
    (1.0, 1.5, "1-1.5"),
    (1.5, 2.0, "1.5-2"),
    (2.0, 2.5, "2-2.5"),
    (2.5, 3.0, "2.5-3"),
    (3.0, 3.5, "3-3.5"),
    (3.5, 4.0, "3.5-4"),
    (4.0, 5.0, "4-5"),
    (5.0, float("inf"), "5+"),
]
MAX_DEG_BINS = [
    (0.0, 3.0, "0-2"),
    (3.0, 5.0, "3-4"),
    (5.0, 7.0, "5-6"),
    (7.0, 9.0, "7-8"),
    (9.0, 11.0, "9-10"),
    (11.0, 13.0, "11-12"),
    (13.0, 17.0, "13-16"),
    (17.0, 25.0, "17-24"),
    (25.0, float("inf"), "25+"),
]

SPECIAL_TOKEN_STRINGS = {"<pad>", "<sos>", "<eos>"}


def parse_checkpoint_name(checkpoint_name, known_pools=None):
    stem = Path(checkpoint_name).stem

    if "_" not in stem:
        raise ValueError(
            f"Checkpoint name '{checkpoint_name}' must follow <layers>_<conv>_<pool>.pt"
        )

    layers_str, remainder = stem.split("_", 1)

    try:
        layers = int(layers_str)
    except ValueError as exc:
        raise ValueError(
            f"Unable to parse layer count from checkpoint name '{checkpoint_name}'"
        ) from exc

    for pool in sorted(known_pools or [], key=len, reverse=True):
        suffix = f"_{pool}"
        if remainder.endswith(suffix):
            conv = remainder[: -len(suffix)]
            if conv:
                return layers, conv, pool

    if "_" not in remainder:
        raise ValueError(
            f"Unable to parse conv/pool from checkpoint name '{checkpoint_name}'"
        )

    conv, pool = remainder.rsplit("_", 1)
    if not conv or not pool:
        raise ValueError(
            f"Unable to parse conv/pool from checkpoint name '{checkpoint_name}'"
        )
    return layers, conv, pool


def list_analysis_checkpoints(pattern, known_pools):
    checkpoints = []
    skipped = []

    for checkpoint_path in sorted(Path(".").glob(pattern)):
        if not checkpoint_path.is_file():
            continue

        try:
            parse_checkpoint_name(checkpoint_path.name, known_pools=known_pools)
            checkpoints.append(checkpoint_path)
        except ValueError:
            skipped.append(checkpoint_path.name)

    return checkpoints, skipped


def load_analysis_model(checkpoint_path, device):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    known_pools = {"add", "mean", "attention", "set2set", "multi_stat"}
    known_pools.update(config["model"].get("pool_types", []))

    layers, conv_type, pool_type = parse_checkpoint_name(
        checkpoint_path.name,
        known_pools=known_pools,
    )

    supported_conv_types = {"gin", "gcn", "gatv2", "dir_gin", "dir_gcn", "dir_gatv2"}
    if conv_type not in supported_conv_types:
        raise ValueError(
            f"Unsupported conv type '{conv_type}' parsed from '{checkpoint_path.name}'"
        )

    encoder = Encoder(
        type_id_vocab_size=node_type_vocab_size,
        attr_id_vocab_size=node_attr_mapping.shape[0],
        max_depth=config["data"]["max_depth"],
        emb_dim=config["model"]["emb_dim"],
        num_layers=layers,
        conv_type=conv_type,
        pool_type=pool_type,
        dropout=config["model"]["dropout"],
        num_heads=config["model"]["num_heads"],
        use_residual=config["model"]["use_residual"],
        use_graph_norm=config["model"]["use_graph_norm"],
    ).to(device)

    decoder = GraphConditionedTransformerDecoder(
        encoder.attr_emb,
        node_attr_mapping.shape[0],
        config["model"]["emb_dim"],
        PAD,
    ).to(device)

    loaded_model = Graph2SeqModel(encoder, decoder).to(device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    loaded_model.load_state_dict(state_dict)
    loaded_model.eval()

    return loaded_model, layers, conv_type, pool_type


def normalize_sequence(seq_like):
    if torch.is_tensor(seq_like):
        raw = seq_like.detach().cpu().view(-1).tolist()
    elif isinstance(seq_like, np.ndarray):
        raw = seq_like.reshape(-1).tolist()
    elif isinstance(seq_like, (list, tuple)):
        raw = list(seq_like)
    else:
        raw = [seq_like]

    tokens = []
    for tok in raw:
        if torch.is_tensor(tok):
            tok = tok.item()

        tok_id = None
        if isinstance(tok, (int, np.integer)):
            tok_id = int(tok)
        elif isinstance(tok, (float, np.floating)) and float(tok).is_integer():
            tok_id = int(tok)

        if tok_id is not None:
            if tok_id in {PAD, SOS}:
                continue
            if tok_id == EOS:
                break
            tok_str = node_idx_to_attr_dict.get(tok_id, "<unk>")
        else:
            tok_str = str(tok)
            if tok_str in {"<sos>", "<pad>"}:
                continue
            if tok_str == "<eos>":
                break

        if tok_str in SPECIAL_TOKEN_STRINGS:
            continue

        tokens.append(tok_str)

    return tokens


def compute_topology_stats(data):
    num_nodes = int(data.num_nodes) if data.num_nodes is not None else 0
    num_edges = int(data.edge_index.size(1)) if data.edge_index is not None else 0

    if num_nodes > 0 and num_edges > 0:
        deg = degree(data.edge_index[0], num_nodes=num_nodes).cpu().numpy()
        avg_degree = float(deg.mean())
        max_degree = float(deg.max())
    else:
        avg_degree = 0.0
        max_degree = 0.0

    if hasattr(data, "node_depth") and data.node_depth is not None and data.node_depth.numel() > 0:
        max_depth = float(data.node_depth.max().item())
    else:
        max_depth = 0.0

    return {
        "num_nodes": num_nodes,
        "num_edges": num_edges,
        "avg_degree": avg_degree,
        "max_degree": max_degree,
        "max_depth": max_depth,
    }


def assign_bin(value, bins):
    for idx, (left, right, label) in enumerate(bins):
        last_bin = idx == len(bins) - 1
        if value >= left and (value < right or (last_bin and value <= right)):
            return label
    return None


def materialize_loader_dataset(loader):
    dataset_obj = getattr(loader, "dataset", None)
    if dataset_obj is None:
        raise ValueError("Loader does not expose a dataset")
    return [dataset_obj[i] for i in range(len(dataset_obj))]


def build_test_split_loaders(base_loader, batch_size, num_workers=0, min_graphs=1):
    graphs = materialize_loader_dataset(base_loader)

    split_specs = {
        "num_nodes": NODE_COUNT_BINS,
        "max_depth": DEPTH_BINS,
        "avg_degree": AVG_DEG_BINS,
        "max_degree": MAX_DEG_BINS,
    }

    grouped_graphs = {split_key: defaultdict(list) for split_key in split_specs}

    for data in tqdm(graphs, desc="Building split bins"):
        stats = compute_topology_stats(data)
        for split_key, bins in split_specs.items():
            label = assign_bin(stats[split_key], bins)
            if label is None:
                continue
            grouped_graphs[split_key][label].append(data)

    split_loaders = {}
    split_rows = []

    for split_key, label_to_graphs in grouped_graphs.items():
        for label, subset in sorted(label_to_graphs.items()):
            if len(subset) < min_graphs:
                continue

            split_name = f"{split_key}:{label}"
            split_loaders[split_name] = DataLoader(
                subset,
                batch_size=batch_size,
                shuffle=False,
                num_workers=num_workers,
            )
            split_rows.append(
                {
                    "split_name": split_name,
                    "split_feature": split_key,
                    "split_bin": label,
                    "num_graphs": len(subset),
                }
            )

    split_df = pd.DataFrame(split_rows).sort_values(["split_feature", "split_bin"]).reset_index(drop=True)
    return split_loaders, split_df


def ogb_f1(evaluator, seq_ref, seq_pred):
    return float(evaluator.eval({"seq_ref": seq_ref, "seq_pred": seq_pred})["F1"])


def collect_error_analysis_data(
    model,
    loader,
    device,
    evaluator,
    max_batches=None,
    max_tokens=20,
):
    """
    For each graph in the loader, collect:
      - official OGB per-graph F1 score
      - structural features: num_nodes, num_edges, avg_degree, max_degree,
        tree_depth (from node_depth), is_connected, num_connected_components,
        branching_factor, fraction of each node type group
      - generation features: predicted length, reference length, length gap
    """
    model.eval()
    records = []
    type_idx_to_name = dict(zip(type_idx_mapping["type idx"], type_idx_mapping["type"]))

    all_seq_ref = []
    all_seq_pred = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(loader, desc="Collecting")):
            if max_batches and batch_idx >= max_batches:
                break

            batch = batch.to(device)
            preds = model.generate(batch, max_tokens=max_tokens)
            y = batch.y

            ptr = batch.ptr.cpu()

            for i in range(batch.num_graphs):
                n_start = ptr[i].item()
                n_end = ptr[i + 1].item()
                num_nodes = n_end - n_start

                x_i = batch.x[n_start:n_end].cpu()
                depth_i = batch.node_depth[n_start:n_end].cpu().squeeze(-1)

                edge_mask = (
                    (batch.edge_index[0] >= n_start)
                    & (batch.edge_index[0] < n_end)
                    & (batch.edge_index[1] >= n_start)
                    & (batch.edge_index[1] < n_end)
                )
                ei_i = batch.edge_index[:, edge_mask].cpu() - n_start
                num_edges = ei_i.size(1)

                deg = degree(ei_i[0], num_nodes=num_nodes).numpy()
                avg_deg = float(deg.mean()) if len(deg) > 0 else 0.0
                max_deg = float(deg.max()) if len(deg) > 0 else 0.0

                max_depth = float(depth_i.max().item()) if num_nodes > 0 else 0.0

                G = nx.DiGraph()
                G.add_nodes_from(range(num_nodes))
                if num_edges > 0:
                    G.add_edges_from(ei_i.T.tolist())
                G_undir = G.to_undirected()
                connected = nx.is_connected(G_undir) if num_nodes > 1 else True
                num_cc = nx.number_connected_components(G_undir)

                out_deg = np.array([d for _, d in G.out_degree()])
                non_leaf_deg = out_deg[out_deg > 0]
                branching = float(non_leaf_deg.mean()) if len(non_leaf_deg) > 0 else 1.0

                type_ids = x_i[:, 0].numpy() if x_i.dim() > 1 else x_i.numpy()
                unique_types, counts = np.unique(type_ids, return_counts=True)
                type_dist = {
                    type_idx_to_name.get(int(t), "unknown"): int(c)
                    for t, c in zip(unique_types, counts)
                }

                structural_types = {
                    "FunctionDef",
                    "ClassDef",
                    "Module",
                    "For",
                    "While",
                    "If",
                    "With",
                    "Try",
                    "ExceptHandler",
                    "AsyncFunctionDef",
                    "AsyncFor",
                    "AsyncWith",
                }
                identifier_types = {"Name", "Attribute", "arg", "Starred"}
                literal_types = {
                    "Num",
                    "Str",
                    "Bytes",
                    "NameConstant",
                    "Constant",
                    "List",
                    "Tuple",
                    "Dict",
                    "Set",
                }

                total = max(num_nodes, 1)
                frac_structural = sum(type_dist.get(t, 0) for t in structural_types) / total
                frac_identifier = sum(type_dist.get(t, 0) for t in identifier_types) / total
                frac_literal = sum(type_dist.get(t, 0) for t in literal_types) / total

                ref_tokens = normalize_sequence(y[i])
                pred_tokens = normalize_sequence(preds[i])

                all_seq_ref.append(ref_tokens)
                all_seq_pred.append(pred_tokens)

                graph_f1 = ogb_f1(evaluator, [ref_tokens], [pred_tokens])

                records.append(
                    {
                        "f1": graph_f1,
                        "num_nodes": num_nodes,
                        "num_edges": num_edges,
                        "avg_degree": avg_deg,
                        "max_degree": max_deg,
                        "max_depth": max_depth,
                        "is_connected": int(connected),
                        "num_components": num_cc,
                        "branching": branching,
                        "frac_structural": frac_structural,
                        "frac_identifier": frac_identifier,
                        "frac_literal": frac_literal,
                        "pred_len": len(pred_tokens),
                        "ref_len": len(ref_tokens),
                        "len_gap": len(pred_tokens) - len(ref_tokens),
                    }
                )

    split_f1 = ogb_f1(evaluator, all_seq_ref, all_seq_pred) if all_seq_ref else float("nan")
    return pd.DataFrame(records), split_f1


def loader_mode_for_conv(conv_type):
    return "directed" if str(conv_type).startswith("dir_") else "undirected"


known_pools = {"add", "mean", "attention", "set2set", "multi_stat"}
known_pools.update(config["model"].get("pool_types", []))

if analysis_checkpoint_names:
    checkpoint_paths = []
    missing_checkpoints = []
    invalid_checkpoints = []

    for checkpoint_name in analysis_checkpoint_names:
        checkpoint_path = Path(checkpoint_name)
        if not checkpoint_path.is_file():
            missing_checkpoints.append(checkpoint_name)
            continue

        try:
            parse_checkpoint_name(checkpoint_path.name, known_pools=known_pools)
            checkpoint_paths.append(checkpoint_path)
        except ValueError:
            invalid_checkpoints.append(checkpoint_name)

    skipped_checkpoints = []

    if missing_checkpoints:
        raise FileNotFoundError(
            "Missing checkpoint files:\n"
            + "\n".join(f"  - {name}" for name in missing_checkpoints)
        )

    if invalid_checkpoints:
        raise ValueError(
            "These checkpoints do not match <layers>_<conv>_<pool>.pt:\n"
            + "\n".join(f"  - {name}" for name in invalid_checkpoints)
        )
else:
    checkpoint_paths, skipped_checkpoints = list_analysis_checkpoints(
        analysis_checkpoint_glob or "*.pt",
        known_pools,
    )

if not checkpoint_paths:
    raise FileNotFoundError(
        "No checkpoints matched the required naming format <layers>_<conv>_<pool>.pt"
    )

split_loaders_by_mode = {}
split_loader_overview_frames = []

for edge_mode, base_test_loader in [
    ("directed", test_loader_directed),
    ("undirected", test_loader_undirected),
]:
    split_loaders_mode, split_overview_mode_df = build_test_split_loaders(
        base_test_loader,
        batch_size=config["train"]["batch_size"],
        num_workers=config["data"]["num_workers"],
        min_graphs=analysis_min_split_graphs,
    )
    split_loaders_by_mode[edge_mode] = split_loaders_mode

    if not split_overview_mode_df.empty:
        split_overview_mode_df = split_overview_mode_df.copy()
        split_overview_mode_df["edge_mode"] = edge_mode
        split_loader_overview_frames.append(split_overview_mode_df)

if not split_loader_overview_frames:
    raise RuntimeError(
        "No split loaders were created. Lower analysis_min_split_graphs or adjust split bins."
    )

split_loader_overview_df = pd.concat(
    split_loader_overview_frames,
    ignore_index=True,
).sort_values(["edge_mode", "split_feature", "split_bin"]).reset_index(drop=True)

print("\nSplit loaders built from test set:")
print(split_loader_overview_df.to_string(index=False))

analysis_evaluator = Evaluator(name=d_name)
analysis_results = {}
analysis_metadata = []

for checkpoint_path in checkpoint_paths:
    model, analysis_layers, analysis_conv, analysis_pool = load_analysis_model(
        checkpoint_path,
        device,
    )

    edge_mode = loader_mode_for_conv(analysis_conv)
    active_split_loaders = split_loaders_by_mode[edge_mode]

    model_tag = checkpoint_path.stem
    print(
        f"\nModel {model_tag}.pt "
        f"(layers={analysis_layers}, conv={analysis_conv}, pool={analysis_pool}, edge_mode={edge_mode})"
    )

    for split_name, split_loader in active_split_loaders.items():
        print(f"  Running split: {split_name}")

        model_df, split_f1 = collect_error_analysis_data(
            model,
            split_loader,
            device,
            evaluator=analysis_evaluator,
            max_tokens=analysis_max_tokens,
        )

        run_tag = f"{model_tag} | {split_name}"
        model_df["model_tag"] = model_tag
        model_df["split_name"] = split_name
        model_df["edge_mode"] = edge_mode
        analysis_results[run_tag] = model_df

        analysis_metadata.append(
            {
                "run_tag": run_tag,
                "model_tag": model_tag,
                "layers": analysis_layers,
                "conv": analysis_conv,
                "pool": analysis_pool,
                "edge_mode": edge_mode,
                "split_name": split_name,
                "num_graphs": len(model_df),
                "split_f1": split_f1,
            }
        )

analysis_overview_df = pd.DataFrame(analysis_metadata).sort_values(
    ["model_tag", "split_name"]
).reset_index(drop=True)
df = pd.concat(analysis_results.values(), ignore_index=True)

overview_csv = analysis_output_dir / "analysis_split_overview.csv"
analysis_overview_df.to_csv(overview_csv, index=False)

if skipped_checkpoints:
    print("\nSkipped files (did not match naming convention):")
    for name in skipped_checkpoints:
        print(f"  - {name}")

print("\nLoaded checkpoints and split-level OGB F1:")
print(analysis_overview_df.to_string(index=False))
print(f"\nCombined rows across all runs: {len(df)}")
print(f"Saved split overview CSV: {overview_csv}")
print(f"Plots will be saved under: {analysis_output_dir.resolve()}")

Building split bins: 100%|██████████| 21948/21948 [00:01<00:00, 11307.65it/s]
/tmp/ipykernel_55/2239188591.py:296: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  split_loaders[split_name] = DataLoader(
Building split bins: 100%|██████████| 21948/21948 [00:01<00:00, 11701.34it/s]
/tmp/ipykernel_55/2239188591.py:296: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  split_loaders[split_name] = DataLoader(



Split loaders built from test set:
       split_name split_feature split_bin  num_graphs  edge_mode
 avg_degree:0.5-1    avg_degree     0.5-1       21948   directed
 max_degree:11-12    max_degree     11-12        1275   directed
 max_degree:13-16    max_degree     13-16        1238   directed
 max_degree:17-24    max_degree     17-24         789   directed
   max_degree:25+    max_degree       25+         362   directed
   max_degree:3-4    max_degree       3-4        4562   directed
   max_degree:5-6    max_degree       5-6        7421   directed
   max_degree:7-8    max_degree       7-8        4124   directed
  max_degree:9-10    max_degree      9-10        2177   directed
  max_depth:10-12     max_depth     10-12        5465   directed
  max_depth:13-15     max_depth     13-15        1044   directed
  max_depth:16-18     max_depth     16-18         136   directed
  max_depth:19-21     max_depth     19-21          24   directed
    max_depth:22+     max_depth       22+          31 

Collecting:   0%|          | 0/102 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(
Collecting: 100%|██████████| 102/102 [00:19<00:00,  5.27it/s]


  Running split: num_nodes:151-200


Collecting:   0%|          | 0/56 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(
Collecting: 100%|██████████| 56/56 [00:11<00:00,  4.91it/s]


...

  Running split: max_degree:7-8


Collecting:   0%|          | 0/179 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(
Collecting: 100%|██████████| 179/179 [00:31<00:00,  5.63it/s]


  Running split: max_degree:9-10


Collecting:   0%|          | 0/96 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(
Collecting: 100%|██████████| 96/96 [00:19<00:00,  5.01it/s]


Loaded checkpoints and split-level OGB F1:
                                  run_tag             model_tag  layers      conv      pool  edge_mode        split_name  num_graphs  split_f1
 8_dir_gatv2_attention | avg_degree:0.5-1 8_dir_gatv2_attention       8 dir_gatv2 attention   directed  avg_degree:0.5-1       21948  0.165444
 8_dir_gatv2_attention | max_degree:11-12 8_dir_gatv2_attention       8 dir_gatv2 attention   directed  max_degree:11-12        1275  0.150422
 8_dir_gatv2_attention | max_degree:13-16 8_dir_gatv2_attention       8 dir_gatv2 attention   directed  max_degree:13-16        1238  0.155863
 8_dir_gatv2_attention | max_degree:17-24 8_dir_gatv2_attention       8 dir_gatv2 attention   directed  max_degree:17-24         789  0.140802
   8_dir_gatv2_attention | max_degree:25+ 8_dir_gatv2_attention       8 dir_gatv2 attention   directed    max_degree:25+         362  0.113141
   8_dir_gatv2_attention | max_degree:3-4 8_dir_gatv2_attention       8 dir_gatv2 attention   dire

In [14]:
from ogb.graphproppred import Evaluator

evaluator = Evaluator(name = "ogbg-code2")

def evaluate_f1(data_loader):
    model.eval()

    seq_ref = []
    seq_pred = []

    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)

            y = batch.y

            # Model predictions
            pred = model.generate(batch, 10)

            for i in range(len(pred)):
                pred_tokens = []
                for j in range(len(pred[i])):
                    pred_tokens.append(
                        node_idx_to_attr_dict[pred[i][j].item()]
                    )
                    if(pred[i][j].item()==EOS):
                        break
                if "<sos>" in pred_tokens:
                  pred_tokens.remove("<sos>")
                if "<eos>" in pred_tokens:
                  pred_tokens.remove("<eos>")
                seq_pred.append(pred_tokens)

            for i in range(len(y)):

                seq_ref.append(y[i])

    input_dict = {
        "seq_ref": seq_ref,
        "seq_pred": seq_pred
    }

    result = evaluator.eval(input_dict)
    return result

In [ ]:
# ── Re-evaluate splits using your evaluate_f1 function ──────────────────────

custom_eval_results = []

for checkpoint_path in checkpoint_paths:
    model, analysis_layers, analysis_conv, analysis_pool = load_analysis_model(
        checkpoint_path, device
    )
    model_tag = checkpoint_path.stem
    edge_mode = loader_mode_for_conv(analysis_conv)
    active_split_loaders = split_loaders_by_mode[edge_mode]
    print(
        f"\nEvaluating {model_tag}  "
        f"(layers={analysis_layers}, conv={analysis_conv}, pool={analysis_pool}, edge_mode={edge_mode})"
    )

    for split_name, split_loader in active_split_loaders.items():
        result = evaluate_f1(split_loader)
        f1_score = result["F1"]

        custom_eval_results.append({
            "model_tag":   model_tag,
            "layers":      analysis_layers,
            "conv":        analysis_conv,
            "pool":        analysis_pool,
            "edge_mode":   edge_mode,
            "split_name":  split_name,
            "num_graphs":  len(split_loader.dataset),
            "custom_f1":   f1_score,
        })

        print(f"  {split_name:40s}  F1={f1_score:.4f}")

custom_eval_df = pd.DataFrame(custom_eval_results).sort_values(
    ["model_tag", "split_name"]
).reset_index(drop=True)

# Save to CSV alongside the other analysis outputs
custom_csv = analysis_output_dir / "custom_eval_split_results.csv"
custom_eval_df.to_csv(custom_csv, index=False)

print("\nCustom-eval results:")
print(custom_eval_df.to_string(index=False))
print(f"\nSaved to: {custom_csv}")


Evaluating 8_dir_gatv2_attention  (layers=8, conv=dir_gatv2, pool=attention, edge_mode=directed)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


  num_nodes:101-150                         F1=0.1628
  num_nodes:151-200                         F1=0.1545
  num_nodes:200+                            F1=0.1296
  num_nodes:21-35                           F1=0.1919
  num_nodes:36-50                           F1=0.1846
  num_nodes:51-75                           F1=0.1755
  num_nodes:76-100                          F1=0.1712
  max_depth:10-12                           F1=0.1443
  max_depth:13-15                           F1=0.1220
  max_depth:16-18                           F1=0.1445
  max_depth:19-21                           F1=0.1139
  max_depth:22+                             F1=0.1028
  max_depth:4-6                             F1=0.1656
  max_depth:7-9                             F1=0.1816
  avg_degree:0.5-1                          F1=0.1654
  max_degree:11-12                          F1=0.1504
  max_degree:13-16                          F1=0.1559
  max_degree:17-24                          F1=0.1408
  max_degree:25+            

In [ ]:
# ── Evaluate on the full test set using your evaluate_f1 function ────────────

for checkpoint_path in checkpoint_paths:
    model, analysis_layers, analysis_conv, analysis_pool = load_analysis_model(
        checkpoint_path, device
    )
    model_tag = checkpoint_path.stem
    edge_mode = loader_mode_for_conv(analysis_conv)
    full_test_loader = test_loader_directed if edge_mode == "directed" else test_loader_undirected
    print(
        f"\nEvaluating {model_tag}  "
        f"(layers={analysis_layers}, conv={analysis_conv}, pool={analysis_pool}, edge_mode={edge_mode})"
    )

    result = evaluate_f1(full_test_loader)
    f1_score = result["F1"]
    print(f"  Full test set F1 = {f1_score:.4f}")